# 01 — Load & QC

Veri yukleme, yapisal butunluk kontrolleri, zaman/sinyal dogrulama, QC bayraklari.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ANALYSIS_ROOT = Path("..").resolve()
sys.path.insert(0, str(ANALYSIS_ROOT))

from src.loader import load_all
from src.qc import (
    check_structural_integrity,
    check_timing,
    check_signals,
    flag_trials,
    add_analysis_mask,
    check_format_regression,
)

with open(ANALYSIS_ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

RAW_DIR = ANALYSIS_ROOT / config["paths"]["raw_dir"]
INTERIM_DIR = ANALYSIS_ROOT / config["paths"]["interim_dir"]
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
print(f"RAW_DIR:     {RAW_DIR}")
print(f"INTERIM_DIR: {INTERIM_DIR}")

## 1. Kesif ve yukleme

In [ ]:
df_samples, df_trials, metadata, session_report = load_all(RAW_DIR)

print(f"Oturumlar: {len(session_report)}")
print(f"  Secilen:   {sum(1 for s in session_report if s['status'] == 'selected')}")
print(f"  Yarim:     {sum(1 for s in session_report if s['status'] == 'incomplete')}")
print(f"
Sample sayisi: {len(df_samples):,}")
print(f"Trial sayisi:  {len(df_trials)}")
print(f"Metadata:      {len(metadata)} dosya")

report_df = pd.DataFrame(session_report)[
    ["participant_id", "session_id", "status",
     "has_metadata", "has_timeseries", "has_trial_summary",
     "measurement_trial_count", "warnings"]
]
report_df

## 2. Yapisal butunluk

In [ ]:
integrity_issues = check_structural_integrity(df_trials, metadata, config)

if integrity_issues:
    df_issues = pd.DataFrame(integrity_issues)
    print(f"{len(integrity_issues)} sorun:")
    display(df_issues)
else:
    print("Yapisal sorun yok.")

## 3. Zaman ve ornekleme

In [ ]:
timing = check_timing(df_samples, config)

problems = timing[
    timing["has_time_reversal"]
    | timing["has_gap"]
    | timing["has_dup_index"]
    | timing["has_skip_index"]
    | (timing["nan_count"] > 0)
    | (timing["angle_out_of_range"] > 0)
]

if len(problems) > 0:
    print(f"{len(problems)} trial'da zaman/ornekleme sorunu:")
    display(problems)
else:
    print("Zaman ve ornekleme sorunsuz.")

print(f"
dt ortalamasi:     {timing['dt_mean'].mean():.6f} s")
print(f"dt std ortalamasi: {timing['dt_std'].mean():.8f} s")
print(f"dt max sapma:      {timing['dt_max_dev'].max():.6f} s")

## 4. Sinyal akil sagligi

In [ ]:
signals = check_signals(df_samples, config)
warn_thr = config["qc"]["velocity_correlation_warn"]

# hiz-pozisyon korelasyonu
low_corr = signals[
    (signals["cart_velocity_corr"] < warn_thr)
    | (signals["omega_velocity_corr"] < warn_thr)
]
if len(low_corr) > 0:
    print(f"{len(low_corr)} trial'da hiz-pozisyon korelasyonu < {warn_thr}:")
    display(low_corr[["participant_id", "trial_id",
                       "cart_velocity_corr", "omega_velocity_corr"]])
else:
    print(f"Hiz-pozisyon korelasyonu tum trial'larda >= {warn_thr}")

# force tutarliligi
force_bad = signals[~signals["force_input_consistent"]]
if len(force_bad) > 0:
    print(f"
{len(force_bad)} trial'da force != input_applied * max_force:")
    display(force_bad[["participant_id", "trial_id"]])
else:
    max_f = config["physics"]["max_force_n"]
    print(f"
Force tutarliligi OK (active: force = input_applied x {max_f} N)")

# phase <-> is_resetting
phase_bad = signals[~signals["phase_reset_consistent"]]
if len(phase_bad) > 0:
    print(f"
{len(phase_bad)} trial'da phase/is_resetting tutarsiz:")
    display(phase_bad[["participant_id", "trial_id"]])
else:
    print("
phase ve is_resetting birebir tutarli.")

## 5. Trial gecerliligi

In [ ]:
df_trials = flag_trials(df_trials, df_samples, config)

n_pass = int(df_trials["qc_pass"].sum())
n_fail = len(df_trials) - n_pass

print(f"Toplam trial: {len(df_trials)}")
print(f"  qc_pass=True:  {n_pass}")
print(f"  qc_pass=False: {n_fail}")

if n_fail > 0:
    print("
Dusen trial'lar:")
    display(
        df_trials[~df_trials["qc_pass"]][
            ["participant_id", "trial_id", "noise_level_id", "practice", "qc_flags"]
        ]
    )

## 6. Sample maskesi

In [ ]:
df_samples = add_analysis_mask(df_samples, df_trials)

n_inc = int(df_samples["analysis_include"].sum())
n_exc = len(df_samples) - n_inc

print(f"Toplam sample:         {len(df_samples):,}")
print(f"  analysis_include:    {n_inc:,}")
print(f"  disarida:            {n_exc:,}")

# dislanma nedenleri (ortusebilir)
print("
Dislanma nedenleri:")
print(f"  phase != active:  {(df_samples['phase'] != 'active').sum():,}")
print(f"  practice trial:   {(df_samples['practice'] != 0).sum():,}")
print(f"  window_focused=0: {(df_samples['window_focused'] != 1).sum():,}")

qc_fail_ids = df_trials.loc[~df_trials["qc_pass"], ["participant_id", "trial_id"]]
if len(qc_fail_ids) > 0:
    n_qc = df_samples.merge(qc_fail_ids, on=["participant_id", "trial_id"]).shape[0]
    print(f"  qc_pass=False:    {n_qc:,}")

# analize giren measurement trial sayisi
meas = df_trials[(df_trials["practice"] == 0) & df_trials["qc_pass"]]
print(f"
Analize giren measurement trial: {len(meas)}")
for pid in sorted(meas["participant_id"].unique()):
    n = int((meas["participant_id"] == pid).sum())
    print(f"  {pid}: {n}")

## 7. Format regresyon takibi

In [ ]:
regression = check_format_regression(metadata, config)

if regression:
    df_reg = pd.DataFrame(regression)
    present = int(df_reg["present"].sum())
    missing = len(df_reg) - present
    print(f"Veri_Kayit_Istekleri.md'den istenen alanlar: {len(df_reg)}")
    print(f"  Mevcut: {present}")
    print(f"  Eksik:  {missing}")
    if missing > 0:
        print("
Eksik alanlar:")
        display(df_reg[~df_reg["present"]])
else:
    print("Metadata dosyasi yok, kontrol yapilamadi.")

## Cikti

In [ ]:
df_samples.to_parquet(INTERIM_DIR / "samples_clean.parquet", index=False)
df_trials.to_parquet(INTERIM_DIR / "trials_clean.parquet", index=False)

print("Kaydedildi:")
print(f"  {INTERIM_DIR / 'samples_clean.parquet'}  ({len(df_samples):,} satir)")
print(f"  {INTERIM_DIR / 'trials_clean.parquet'}  ({len(df_trials)} satir)")